# Gold — Perfil de consumo por município

Desenvolvido por: Ygor Moraes

Este notebook cria a Gold de perfil de consumo dos clientes por município.

Fontes:
- Silver `ecommerce_pedidos`
- Silver `ecommerce_enderecos`

Regra:
- considerar apenas pedidos com status `ENTREGUE`;
- cruzar o pedido com o endereço de entrega;
- considerar apenas registros com cidade e estado preenchidos;
- agregar consumo por estado e cidade.

Destino ADLS:
- `gold/ecommerce_enderecos/perfil_consumo_municipio`

Destino SQL Server:
- `squad3.gold_ecommerce_enderecos_perfil_consumo_municipio`

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# Define imports, caminhos e parâmetros da Gold.

from pyspark.sql.functions import (
    col,
    countDistinct,
    current_timestamp,
    round as spark_round,
    sum as spark_sum,
    avg,
    trim,
    upper
)

SILVER_PEDIDOS_TABLE = "ecommerce_pedidos"
SILVER_ENDERECOS_TABLE = "ecommerce_enderecos"

SILVER_PEDIDOS_PATH = f"{SILVER_BASE_PATH}{SILVER_PEDIDOS_TABLE}"
SILVER_ENDERECOS_PATH = f"{SILVER_BASE_PATH}{SILVER_ENDERECOS_TABLE}"

GOLD_DOMAIN = "ecommerce_enderecos"
GOLD_KPI = "perfil_consumo_municipio"

GOLD_PATH = f"{GOLD_BASE_PATH}{GOLD_DOMAIN}/{GOLD_KPI}"

FINAL_TABLE_NAME = f"gold_{GOLD_DOMAIN}_{GOLD_KPI}"
FINAL_TABLE = f"{TARGET_SCHEMA}.{FINAL_TABLE_NAME}"

GOLD_WRITE_MODE = "overwrite"
STATUS_PEDIDO_CONSIDERADO = "ENTREGUE"

PEDIDOS_REQUIRED_COLUMNS = [
    "id_pedido",
    "id_cliente",
    "id_endereco_entrega",
    "status_pedido",
    "valor_total",
    "valor_frete"
]

ENDERECOS_REQUIRED_COLUMNS = [
    "id_endereco",
    "cidade",
    "estado"
]

GOLD_KEY_COLUMNS = [
    "estado",
    "cidade"
]

adls_options = get_adls_options()

print("Parâmetros definidos com sucesso.")
print("SILVER_PEDIDOS_PATH:", SILVER_PEDIDOS_PATH)
print("SILVER_ENDERECOS_PATH:", SILVER_ENDERECOS_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("FINAL_TABLE:", FINAL_TABLE)

In [0]:
# Lê as Silvers de pedidos e endereços.

df_pedidos = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_PEDIDOS_PATH)
)

df_enderecos = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_ENDERECOS_PATH)
)

validate_required_columns(df_pedidos, PEDIDOS_REQUIRED_COLUMNS)
validate_required_columns(df_enderecos, ENDERECOS_REQUIRED_COLUMNS)

print("Silver de pedidos lida com sucesso.")
print(f"Total de pedidos: {df_pedidos.count()}")

print("Silver de endereços lida com sucesso.")
print(f"Total de endereços: {df_enderecos.count()}")

In [0]:
# Prepara pedidos e endereços para o join.

df_pedidos_base = (
    df_pedidos
    .select(
        col("id_pedido"),
        col("id_cliente").cast("int").alias("id_cliente"),
        col("id_endereco_entrega").cast("int").alias("id_endereco_entrega"),
        upper(trim(col("status_pedido"))).alias("status_pedido"),
        col("valor_total"),
        col("valor_frete")
    )
    .dropDuplicates(["id_pedido"])
)

df_enderecos_base = (
    df_enderecos
    .select(
        col("id_endereco"),
        upper(trim(col("cidade"))).alias("cidade"),
        upper(trim(col("estado"))).alias("estado")
    )
    .dropDuplicates(["id_endereco"])
)

print("Bases preparadas.")
print(f"Pedidos distintos: {df_pedidos_base.count()}")
print(f"Endereços distintos: {df_enderecos_base.count()}")

In [0]:
# Cruza pedidos entregues com o endereço de entrega.

df_consumo_base = (
    df_pedidos_base
    .filter(col("status_pedido") == STATUS_PEDIDO_CONSIDERADO)
    .join(
        df_enderecos_base,
        df_pedidos_base["id_endereco_entrega"] == df_enderecos_base["id_endereco"],
        "left"
    )
)

df_consumo_municipio = (
    df_consumo_base
    .filter(col("estado").isNotNull() & col("cidade").isNotNull())
)

total_pedidos_entregues = df_consumo_base.select("id_pedido").distinct().count()
total_pedidos_com_municipio = df_consumo_municipio.select("id_pedido").distinct().count()
total_pedidos_sem_municipio = total_pedidos_entregues - total_pedidos_com_municipio

print("Join realizado com sucesso.")
print(f"Pedidos entregues: {total_pedidos_entregues}")
print(f"Pedidos entregues com município válido: {total_pedidos_com_municipio}")
print(f"Pedidos entregues sem município válido: {total_pedidos_sem_municipio}")

In [0]:
# Agrega perfil de consumo por estado e cidade.

df_gold = (
    df_consumo_municipio
    .groupBy("estado", "cidade")
    .agg(
        countDistinct("id_cliente").alias("qtd_clientes_com_pedido"),
        countDistinct("id_pedido").alias("qtd_pedidos_entregues"),
        spark_sum("valor_total").alias("receita_total"),
        spark_sum("valor_frete").alias("frete_total"),
        avg("valor_total").alias("ticket_medio_pedido"),
        avg("valor_frete").alias("frete_medio_pedido")
    )
    .withColumn(
        "consumo_medio_por_cliente",
        spark_round(col("receita_total") / col("qtd_clientes_com_pedido"), 2)
    )
    .withColumn("receita_total", spark_round(col("receita_total"), 2))
    .withColumn("frete_total", spark_round(col("frete_total"), 2))
    .withColumn("ticket_medio_pedido", spark_round(col("ticket_medio_pedido"), 2))
    .withColumn("frete_medio_pedido", spark_round(col("frete_medio_pedido"), 2))
    .withColumn("gold_processed_at", current_timestamp())
    .orderBy("estado", "cidade")
)

print("Gold agregada com sucesso.")
print(f"Total de linhas Gold: {df_gold.count()}")

display(df_gold)

In [0]:
# Valida chave da Gold e consistência de pedidos e receita.

total_linhas_gold = df_gold.count()

cidades_distintas_gold = (
    df_gold
    .select(*GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

cidades_duplicadas = total_linhas_gold - cidades_distintas_gold

validacao_gold = (
    df_gold
    .agg(
        spark_sum("qtd_pedidos_entregues").alias("total_pedidos_gold"),
        spark_round(spark_sum("receita_total"), 2).alias("receita_gold")
    )
    .collect()[0]
)

receita_base = (
    df_consumo_municipio
    .agg(
        spark_round(spark_sum("valor_total"), 2).alias("receita_base")
    )
    .collect()[0]["receita_base"]
)

print(f"Total de linhas Gold: {total_linhas_gold}")
print(f"Cidades duplicadas: {cidades_duplicadas}")
print(f"Pedidos com município válido: {total_pedidos_com_municipio}")
print(f"Pedidos Gold: {validacao_gold['total_pedidos_gold']}")
print(f"Receita base: {receita_base}")
print(f"Receita Gold: {validacao_gold['receita_gold']}")
print(f"Pedidos entregues sem município válido: {total_pedidos_sem_municipio}")

if cidades_duplicadas != 0:
    raise ValueError("Validação falhou: existem cidades duplicadas na Gold.")

if validacao_gold["total_pedidos_gold"] != total_pedidos_com_municipio:
    raise ValueError("Validação falhou: total de pedidos da Gold diferente da base com município.")

if validacao_gold["receita_gold"] != receita_base:
    raise ValueError("Validação falhou: receita da Gold diferente da base com município.")

print("Validação OK: Gold sem duplicidade e com pedidos/receita consistentes.")

## Decisão de recorte da Gold

Esta Gold calcula perfil de consumo por município.

Foram considerados apenas pedidos com status `ENTREGUE` e endereço de entrega com `estado` e `cidade` preenchidos.

Pedidos entregues sem município válido foram mantidos fora da agregação, pois não é seguro atribuí-los a uma cidade.

In [0]:
# Grava a Gold em Delta no ADLS.

(
    df_gold
    .write
    .format("delta")
    .options(**adls_options)
    .option("overwriteSchema", "true")
    .mode(GOLD_WRITE_MODE)
    .save(GOLD_PATH)
)

print("Gold gravada com sucesso no ADLS.")
print("Caminho:", GOLD_PATH)

In [0]:
# Lê a Gold gravada no ADLS.

df_gold_gravada = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(GOLD_PATH)
)

display(df_gold_gravada)

In [0]:
# Valida a Gold gravada no ADLS.

total_linhas_gold_gravada = df_gold_gravada.count()

cidades_distintas_gold_gravada = (
    df_gold_gravada
    .select(*GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

cidades_duplicadas_gold_gravada = total_linhas_gold_gravada - cidades_distintas_gold_gravada

validacao_gold_gravada = (
    df_gold_gravada
    .agg(
        spark_sum("qtd_pedidos_entregues").alias("total_pedidos_gold"),
        spark_round(spark_sum("receita_total"), 2).alias("receita_gold")
    )
    .collect()[0]
)

print(f"Total de linhas Gold gravada: {total_linhas_gold_gravada}")
print(f"Cidades duplicadas Gold gravada: {cidades_duplicadas_gold_gravada}")
print(f"Pedidos Gold gravada: {validacao_gold_gravada['total_pedidos_gold']}")
print(f"Receita Gold gravada: {validacao_gold_gravada['receita_gold']}")

if cidades_duplicadas_gold_gravada != 0:
    raise ValueError("Validação falhou: existem cidades duplicadas na Gold gravada.")

if validacao_gold_gravada["total_pedidos_gold"] != total_pedidos_com_municipio:
    raise ValueError("Validação falhou: total de pedidos da Gold gravada diferente da base com município.")

if validacao_gold_gravada["receita_gold"] != receita_base:
    raise ValueError("Validação falhou: receita da Gold gravada diferente da base com município.")

print("Validação OK: Gold gravada corretamente no ADLS.")

In [0]:
# Prepara a Gold para gravação no SQL Server.

df_gold_sql = df_gold_gravada.select(
    "estado",
    "cidade",
    "qtd_clientes_com_pedido",
    "qtd_pedidos_entregues",
    "receita_total",
    "frete_total",
    "ticket_medio_pedido",
    "frete_medio_pedido",
    "consumo_medio_por_cliente",
    "gold_processed_at"
)

display(df_gold_sql.limit(10))

In [0]:
# Grava a Gold no SQL Server.

write_sql_table(
    df=df_gold_sql,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=FINAL_TABLE,
    mode="overwrite",
    sql_port=SQL_PORT
)

print("Gold gravada com sucesso no SQL Server.")
print("Tabela:", FINAL_TABLE)

In [0]:
# Lê e valida a Gold gravada no SQL Server.

df_gold_sql_gravada = read_sql_table(
    spark=spark,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=FINAL_TABLE,
    sql_port=SQL_PORT
)

total_linhas_sql = df_gold_sql_gravada.count()

cidades_distintas_sql = (
    df_gold_sql_gravada
    .select(*GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

cidades_duplicadas_sql = total_linhas_sql - cidades_distintas_sql

validacao_sql = (
    df_gold_sql_gravada
    .agg(
        spark_sum("qtd_pedidos_entregues").alias("total_pedidos_sql"),
        spark_round(spark_sum("receita_total"), 2).alias("receita_sql")
    )
    .collect()[0]
)

print(f"Total de linhas SQL Server: {total_linhas_sql}")
print(f"Cidades duplicadas SQL Server: {cidades_duplicadas_sql}")
print(f"Pedidos SQL Server: {validacao_sql['total_pedidos_sql']}")
print(f"Receita SQL Server: {validacao_sql['receita_sql']}")

if cidades_duplicadas_sql != 0:
    raise ValueError("Validação falhou: existem cidades duplicadas no SQL Server.")

if validacao_sql["total_pedidos_sql"] != total_pedidos_com_municipio:
    raise ValueError("Validação falhou: total de pedidos no SQL Server diferente da Gold.")

if validacao_sql["receita_sql"] != receita_base:
    raise ValueError("Validação falhou: receita no SQL Server diferente da Gold.")

print("Validação OK: Gold gravada corretamente no SQL Server.")